# Matrix Spike (MS) Time-Series / Drift Analysis

## Purpose

This notebook checks whether Matrix Spike (MS) results show **sustained movement away from their supplied target over time**.

The historical analysis shows how far individual results are from target. This analysis focuses on whether that behaviour continues across multiple observations, which may indicate **suppressed or inflated recovery**.

### Interpretation

- `STANDARD_STATUS` remains the original CCLAS classification.
- The time-series analysis looks for patterns across consecutive MS results.
- A drift flag is a **candidate for analyst review**, not a confirmed failure.

Results are scaled relative to their supplied warning limits:

- **0** = on target
- **+1** = upper warning limit
- **−1** = lower warning limit

A possible drift is flagged when the rolling behaviour moves meaningfully away from the target and remains there over several observations.

The drift settings used here are **POC parameters**, not official Datamine acceptance rules.

In [83]:
# 1. Libraries
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Drift analysis settings

The following settings are used to identify sustained changes in Matrix Spike behaviour over time.

- `ROLLING_WINDOW = 10` — looks at the most recent 10 comparable MS results when calculating the rolling behaviour.

- `MIN_PERIODS = 5` — requires at least 5 results before a rolling value is calculated.

- `EARLY_DRIFT_FRACTION = 0.75` — marks an early drift condition when the rolling behaviour moves at least 75% of the way from the target toward the relevant warning limit.

- `PERSISTENCE_WINDOWS = 4` — requires the early drift condition to continue across at least 4 consecutive rolling windows.

- `STRONG_DRIFT_FRACTION = 1.00` — marks a strong drift condition when the rolling behaviour reaches or exceeds the supplied warning limit.

- `STRONG_PERSISTENCE_WINDOWS = 3` — requires the strong drift condition to continue across at least 3 consecutive rolling windows.

These are configurable POC settings, not existing CCLAS acceptance rules. They are used to reduce the effect of isolated unusual results and focus on sustained behaviour that may require analyst review.

In [84]:
# 2. Configuration
ROLLING_WINDOW = 10
MIN_PERIODS = 5

EARLY_DRIFT_FRACTION = 0.75
PERSISTENCE_WINDOWS = 4

STRONG_DRIFT_FRACTION = 1.00
STRONG_PERSISTENCE_WINDOWS = 3

WORKBOOK_PATH = None
SHEET_NAME = "SPK(MS) Assessment"

print({
    "rolling_window": ROLLING_WINDOW,
    "min_periods": MIN_PERIODS,
    "early_drift_fraction": EARLY_DRIFT_FRACTION,
    "persistence_windows": PERSISTENCE_WINDOWS,
    "strong_drift_fraction": STRONG_DRIFT_FRACTION,
    "strong_persistence_windows": STRONG_PERSISTENCE_WINDOWS,
})

{'rolling_window': 10, 'min_periods': 5, 'early_drift_fraction': 0.75, 'persistence_windows': 4, 'strong_drift_fraction': 1.0, 'strong_persistence_windows': 3}


## 3. Load the Matrix Spike data

The QC workbook is located automatically and the `SPK(MS) Assessment` worksheet is loaded for the drift analysis.

This ensures that only the Matrix Spike assessment data is used in the following steps.

In [85]:
# 3. Locate workbook and load the Matrix Spike sheet
def find_workbook():
    if WORKBOOK_PATH is not None:
        p = Path(WORKBOOK_PATH)
        if not p.exists():
            raise FileNotFoundError(f"WORKBOOK_PATH does not exist: {p}")
        return p

    roots = [Path.cwd(), Path.cwd().parent, Path("/mnt/data")]
    patterns = [
        "*QC*Anomaly*Training*Data*.xlsx",
        "*QC*Anomaly*.xlsx",
        "*.xlsx",
    ]

    candidates = []
    for root in roots:
        if root.exists():
            for pattern in patterns:
                candidates.extend(root.rglob(pattern))

    candidates = [
        p for p in candidates
        if not p.name.startswith("~$")
    ]

    if not candidates:
        raise FileNotFoundError(
            "No workbook was found. Set WORKBOOK_PATH to the QC workbook."
        )

    candidates = list(dict.fromkeys(candidates))
    candidates.sort(
        key=lambda p: (
            "training" not in p.name.lower(),
            "anomaly" not in p.name.lower(),
            len(str(p))
        )
    )
    return candidates[0]


workbook = find_workbook()
print("Workbook:", workbook)

xls = pd.ExcelFile(workbook)

if SHEET_NAME not in xls.sheet_names:
    raise KeyError(
        f"Sheet '{SHEET_NAME}' was not found. Available sheets: {xls.sheet_names}"
    )

df_raw = pd.read_excel(workbook, sheet_name=SHEET_NAME)

print("Selected sheet:", SHEET_NAME)
print("Loaded shape:", df_raw.shape)
display(df_raw.head())

Workbook: /Users/shivanshi/Desktop/QUT-Capstone-Team-32-Datamine-QC-Quality-Control-1/data/raw/QC Anomaly Training Data v2 (1).xlsx


Selected sheet: SPK(MS) Assessment
Loaded shape: (6958, 28)


,ANALYTICAL_TYPE,QC_TYPE,STD_LOT_CODE,STD_CODE,JOB_CODE,NUMERIC_FINAL_VALUE,ANALYSED_DATE,SCHEME_CODE,ANALYTE_CODE,STANDARD_STATUS,PRECISION_STATUS,INTERNAL_MIN_VALUE,INTERNAL_MAX_VALUE,INTERNAL_MIN_INCLUSIVE,INTERNAL_MAX_INCLUSIVE,INTERNAL_MAX_WARNING_VALUE,INTERNAL_MIN_WARNING_VALUE,INTERNAL_MIN_WARNING_INCLUSIVE,INTERNAL_MAX_WARNING_INCLUSIVE,LIM_REP_VALUE,STAT_DL_VALUE,LIM_REP_DUP_VALUE,STAT_DL_DUP_VALUE,INTERNAL_TARGET_VALUE,PARENT_NUMERIC_FINAL_VALUE,UNIT_CODE,SPECIFICATION_CODE,INSTRUMENT_ID
0,Spike,MS,OREAS_502C,OREAS_502C,TSV_LB0010007372,0.093486,2021-06-08 16:35:31,GE_IMS40Q12,IN,Pass,NaN,0.0301,0.1479,Y,Y,0.126362,0.051638,Y,Y,10,0.05,15.0,0.05,0.089,NaN,MG_KG,OREAS_502C,NaN
1,Spike,MS,OREAS_502C,OREAS_502C,TSV_LB0009017856,0.100020,2021-05-10 15:11:04,GE_IMS40Q12,IN,Pass,NaN,0.0301,0.1479,Y,Y,0.126362,0.051638,Y,Y,10,0.05,15.0,0.05,0.089,NaN,MG_KG,OREAS_502C,NaN
2,Spike,MS,OREAS_502C,OREAS_502C,TSV_LB0009017856,0.103217,2021-05-07 12:18:32,GE_IMS40Q12,IN,Pass,NaN,0.0301,0.1479,Y,Y,0.126362,0.051638,Y,Y,10,0.05,15.0,0.05,0.089,NaN,MG_KG,OREAS_502C,NaN
3,Spike,MS,OREAS_502C,OREAS_502C,TSV_LB0009017856,0.062144,2021-05-05 10:31:07,GE_IMS40Q12,IN,Pass,NaN,0.0301,0.1479,Y,Y,0.126362,0.051638,Y,Y,10,0.05,15.0,0.05,0.089,NaN,MG_KG,OREAS_502C,NaN
4,Spike,MS,OREAS_502C,OREAS_502C,TSV_LB0006998356,0.091200,2021-03-03 13:04:39,GE_IMS40Q12,IN,Pass,NaN,0.0301,0.1479,Y,Y,0.126362,0.051638,Y,Y,10,0.05,15.0,0.05,0.089,NaN,MG_KG,OREAS_502C,NaN


## 4. Prepare the MS data for drift analysis

To detect historical drift, each Matrix Spike result is first centred around its supplied internal target.

$$
\text{Deviation}_t
=
\text{MS Result}_t
-
\text{Internal Target}_t
$$

This creates a target-relative time series:

- **0** = result is exactly on target
- **positive deviation** = result is above target
- **negative deviation** = result is below target

The supplied warning and failure limits are also converted to the same target-relative scale so the historical behaviour can be interpreted against the existing QC limits.

The results are then ordered by analysis date within each comparable Scheme–Analyte–Unit group.

This prepares the MS results for the later rolling time-series analysis.

In [86]:
# 4. Prepare the MS data for drift analysis

ms = df_raw.copy()

# Keep only usable Matrix Spike rows
ms["ANALYSED_DATE"] = pd.to_datetime(ms["ANALYSED_DATE"], errors="coerce")

numeric_cols = [
    "NUMERIC_FINAL_VALUE",
    "INTERNAL_TARGET_VALUE",
    "INTERNAL_MIN_VALUE",
    "INTERNAL_MAX_VALUE",
    "INTERNAL_MIN_WARNING_VALUE",
    "INTERNAL_MAX_WARNING_VALUE",
]

for col in numeric_cols:
    ms[col] = pd.to_numeric(ms[col], errors="coerce")

required_cols = [
    "ANALYSED_DATE",
    "SCHEME_CODE",
    "ANALYTE_CODE",
    "UNIT_CODE",
    "NUMERIC_FINAL_VALUE",
    "INTERNAL_TARGET_VALUE",
]

ms = ms.dropna(subset=required_cols).copy()

# Centre each MS result around its supplied target
ms["DEVIATION"] = (
    ms["NUMERIC_FINAL_VALUE"]
    - ms["INTERNAL_TARGET_VALUE"]
)

# Convert the supplied warning/failure limits to the same target-relative scale
ms["LOWER_WARNING_DEV"] = (
    ms["INTERNAL_MIN_WARNING_VALUE"]
    - ms["INTERNAL_TARGET_VALUE"]
)

ms["UPPER_WARNING_DEV"] = (
    ms["INTERNAL_MAX_WARNING_VALUE"]
    - ms["INTERNAL_TARGET_VALUE"]
)

ms["LOWER_FAILURE_DEV"] = (
    ms["INTERNAL_MIN_VALUE"]
    - ms["INTERNAL_TARGET_VALUE"]
)

ms["UPPER_FAILURE_DEV"] = (
    ms["INTERNAL_MAX_VALUE"]
    - ms["INTERNAL_TARGET_VALUE"]
)

# Order comparable MS results chronologically
group_cols = [
    "SCHEME_CODE",
    "ANALYTE_CODE",
    "UNIT_CODE",
]

ms = ms.sort_values(
    group_cols + ["ANALYSED_DATE"]
).reset_index(drop=True)

print("Usable MS rows:", len(ms))

display(
    ms[
        group_cols
        + [
            "ANALYSED_DATE",
            "NUMERIC_FINAL_VALUE",
            "INTERNAL_TARGET_VALUE",
            "DEVIATION",
            "LOWER_WARNING_DEV",
            "UPPER_WARNING_DEV",
            "STANDARD_STATUS",
        ]
    ].head(10)
)

Usable MS rows: 6872


,SCHEME_CODE,ANALYTE_CODE,UNIT_CODE,ANALYSED_DATE,NUMERIC_FINAL_VALUE,INTERNAL_TARGET_VALUE,DEVIATION,LOWER_WARNING_DEV,UPPER_WARNING_DEV,STANDARD_STATUS
0,GE_ICP40Q12,AG,MG_KG,2020-06-29 11:14:08,1.234450,0.779,0.455450,-0.842255,0.842255,Pass
1,GE_ICP40Q12,AG,MG_KG,2020-07-08 05:53:05,1.543689,0.779,0.764689,-0.842255,0.842255,Pass
2,GE_ICP40Q12,AG,MG_KG,2020-07-08 06:49:31,1.650000,0.779,0.871000,-0.842255,0.842255,UpperWarning
3,GE_ICP40Q12,AG,MG_KG,2020-07-09 12:36:17,1.275510,0.779,0.496510,-0.842255,0.842255,Pass
4,GE_ICP40Q12,AG,MG_KG,2020-07-10 03:22:44,1.661376,0.779,0.882376,-0.842255,0.842255,UpperWarning
5,GE_ICP40Q12,AG,MG_KG,2020-07-13 13:54:29,1.160000,0.779,0.381000,-0.842255,0.842255,Pass
6,GE_ICP40Q12,AG,MG_KG,2020-07-14 01:37:45,0.720000,0.779,-0.059000,-0.842255,0.842255,Pass
7,GE_ICP40Q12,AG,MG_KG,2020-07-15 11:56:22,1.341176,0.779,0.562176,-0.842255,0.842255,Pass
8,GE_ICP40Q12,AG,MG_KG,2020-07-17 12:24:41,1.248731,0.779,0.469731,-0.842255,0.842255,Pass
9,GE_ICP40Q12,AG,MG_KG,2020-07-20 05:29:35,1.428571,0.779,0.649571,-0.842255,0.842255,Pass


## 5. Standardise the deviation relative to the supplied warning limits

Because different analytes can have different targets and warning ranges, the raw deviation values are not directly comparable across all MS groups.

To make the historical behaviour comparable, each deviation is scaled relative to the distance between the target and the relevant supplied warning limit.

### Positive deviations

For results above the target:

$$
\text{Warning-Scale Position}
=
\frac{\text{Deviation}}
{\text{Upper Warning Limit} - \text{Target}}
$$

### Negative deviations

For results below the target:

$$
\text{Warning-Scale Position}
=
\frac{\text{Deviation}}
{\text{Target} - \text{Lower Warning Limit}}
$$

### Interpretation

- **0** = exactly on target
- **+0.5** = halfway from the target to the upper warning limit
- **+1** = at the upper warning limit
- **−0.5** = halfway from the target to the lower warning limit
- **−1** = at the lower warning limit

This allows the drift analysis to measure movement relative to each analyte's own supplied QC warning range rather than relying only on the raw deviation size.

## 6. Rolling target-relative behaviour

A rolling mean is calculated **within each Scheme–Analyte–Unit group**.

Two rolling series are kept:

1. `ROLLING_DEVIATION` — easy to interpret in the original measurement units.
2. `ROLLING_WARNING_POSITION` — used for drift logic because it accounts for analyte-specific supplied limits.

A single extreme point can move the raw series, but a sustained rolling displacement requires several observations to support the pattern.
